# RAG using llamaindex

We start with simple LLM prompt using llamaindex 

In [70]:
from dotenv import load_dotenv, find_dotenv
from llama_index.llms.google_genai import GoogleGenAI

load_dotenv(find_dotenv(), override=True)

# Due to Jupyter notebook's event loop, we define this function to execute llm because only one event loop can be executed at once
async def run_llm(llm_instance: GoogleGenAI,message: str) -> str:
    response = await llm_instance.acomplete(message)
    return response

llm = GoogleGenAI(model="gemini-3-flash-preview")
resp = await run_llm(llm, "Hey gemini!")
print(resp)

Hello! How can I help you today?


### Initialize the embeddings

In [74]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

embed_model = GoogleGenAIEmbedding(model="models/text-embedding-004")


### Loading and splitting

In [80]:
from llama_index.readers.web import BeautifulSoupWebReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import  Settings

#Setting global variables for llamaindex functions
Settings.llm = llm
Settings.embed_model = embed_model

document_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

loader = BeautifulSoupWebReader()
documents = loader.load_data(urls=[document_url])

splitter = SentenceSplitter(chunk_size=1000, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(documents)

# Filter empty nodes
nodes = [node for node in nodes if node.get_content().strip()]

### Initialize vector store

In [82]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model=embed_model, insert_batch_size=1)

### Retrieval and generation

In [92]:
from llama_index.core import PromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

retriever = index.as_retriever(similarity_top_k=4)

template_str = """\
You are an expert assistant.
Answer only from the context below; if the answer is not there, say "I don't know."

Context:
---------------------
{context_str}
---------------------

Question: {query_str}
Answer:"""

text_qa_template = PromptTemplate(template_str)

from llama_index.core.response_synthesizers import CompactAndRefine

response_synthesizer = CompactAndRefine(
    text_qa_template=text_qa_template,
    llm=llm,
)

from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

question = "What is Task Decomposition?"

async def run_query(query_engine: RetrieverQueryEngine,query: str) -> str:
    result = await query_engine.aquery(question)
    return result.response

result = await run_query(query_engine, question)

print(result)

Task decomposition is the process of breaking down a complex task into smaller and simpler steps or subgoals to make it more manageable. 

According to the context, it can be achieved through several methods:
*   **Chain of Thought (CoT):** A prompting technique where the model is instructed to "think step by step" to decompose hard tasks.
*   **Tree of Thoughts:** An extension of CoT that explores multiple reasoning possibilities at each step to create a tree structure.
*   **LLM Prompting:** Using simple prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?"
*   **Task-specific instructions:** For example, "Write a story outline" for the task of writing a novel.
*   **Human inputs.**
*   **LLM+P:** An approach that utilizes an external classical planner and Planning Domain Definition Language (PDDL) to generate a plan.
